# Medical Cost Prediction - Machine Learning (Fixed)
## Train and Save Models for Web App

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GridSearchCV
import joblib
import warnings
warnings.filterwarnings('ignore')

# Load data
medical_df = pd.read_csv('medical_insurance.csv')
print('Dataset shape:', medical_df.shape)
print(medical_df.head())

# Categorical encoding
medical_df['sex'] = medical_df['sex'].map({'female':0, 'male':1})
medical_df['smoker'] = medical_df['smoker'].map({'no':0, 'yes':1})
medical_df['region'] = medical_df['region'].map({
    'southeast':1, 'southwest':2, 'northwest':3, 'northeast':4
})

# Split
X = medical_df.drop('charges', axis=1)
y = medical_df['charges']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Scale
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print('Train:', X_train_scaled.shape, 'Test:', X_test_scaled.shape)

In [ ]:
# Linear Regression
lr = LinearRegression()
lr.fit(X_train_scaled, y_train)
lr_pred = lr.predict(X_test_scaled)
print('LR R2:', r2_score(y_test, lr_pred))
lr_cv = cross_val_score(lr, X_train_scaled, y_train, cv=5)
print('LR CV R2:', lr_cv.mean())

# Random Forest with tuning
rf = RandomForestRegressor(random_state=42)
param_grid = {'n_estimators': [100, 200], 'max_depth': [10, None]}
grid = GridSearchCV(rf, param_grid, cv=5, n_jobs=-1)
grid.fit(X_train_scaled, y_train)
rf = grid.best_estimator_
rf_pred = rf.predict(X_test_scaled)
print('RF R2:', r2_score(y_test, rf_pred))
rf_cv = cross_val_score(rf, X_train_scaled, y_train, cv=5)
print('RF CV R2:', rf_cv.mean())
print('Best params:', grid.best_params_)

In [ ]:
# Save models
joblib.dump(lr, 'lr_model.pkl')
joblib.dump(rf, 'rf_model.pkl')
joblib.dump(scaler, 'scaler.pkl')
print('Models saved! Copy to webapp/models/')

In [ ]:
# Colorful plots
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_palette('husl')

fig, axes = plt.subplots(2, 2, figsize=(15, 12))

# 1. Age vs Charges
sns.scatterplot(ax=axes[0,0], x='age', y='charges', hue='smoker', data=medical_df, palette='coolwarm')

# 2. BMI vs Charges
sns.scatterplot(ax=axes[0,1], x='bmi', y='charges', hue='region', data=medical_df, palette='Set2')

# 3. Boxplot smoker
sns.boxplot(ax=axes[1,0], x='smoker', y='charges', data=medical_df, palette='viridis')

# 4. Correlation heatmap
sns.heatmap(medical_df.corr(), annot=True, fmt='.2f', cmap='RdYlBu_r', center=0, ax=axes[1,1])

plt.tight_layout()
plt.show()